## GazeOnlyGAT: components overview

In [1]:
import only_gaze_confs as conf

/store/alutfi/py_venvs/gat2/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


choosing least busy GPU...
running on cuda:1


With the gaze patch features and encodings aggregated into graph nodes, we can feed it through our network. We'll go through each component.

Fetch sample batch

In [2]:
import torch
from gaze_datasets.reflacx import ReflacxGazePatchDataset
from gaze_datasets.dataset_tools import gaze_patch_dataset_collate
from feature_extraction.extractors import GazePatchFeatureExtractor
from node_encodings.sequence import GazeSinCosPosEncoding

gaze_dataset_pth = conf.reflacx_gaze_dataset_pth
desired_label = 'Atelectasis'
ds = ReflacxGazePatchDataset(gaze_dataset_pth, desired_label)
splits, fixations, patches, labels = gaze_patch_dataset_collate((ds[33], ds[34], ds[0]))

feat_ext = GazePatchFeatureExtractor(conf.fe_tail_feat_ext,
                                     conf.fe_state_dict_pth,
                                     conf.fe_channels,
                                     conf.fe_act,
                                     conf.fe_norm,
                                     conf.fe_bias,
                                     conf.fe_w_init_f,
                                     conf.device)
feats = feat_ext(patches.to(conf.device))

max_len = conf.reflacx_max_len
seq_encoding_featcount = conf.seq_encoding_feat_count

seq_encoding = GazeSinCosPosEncoding(max_len, seq_encoding_featcount)
seqs = seq_encoding(splits, adjust_dim=True).to(conf.device)
fixations = fixations.to(conf.device)

nodes = torch.cat([fixations, seqs, feats], 1).float()
nodes.shape, sum([x.shape[1]for x in (seqs, fixations, feats)])

(torch.Size([255, 147, 1, 1]), 147)

In [3]:
ns = nodes.squeeze(-1).squeeze(-1).permute((1, 0))

### GAT Layer
<a id="gat_layer"></a>

The first component is the Graph Attention Convolution Layer.

In [44]:
from model.gat_conv import CausalGATConv

set number of attention heads for the layer

In [45]:
att_heads = 4

In [46]:
gcnn = CausalGATConv(ns.shape[0],
                     ns.shape[0],
                     att_heads,
                     device=conf.device)

In [47]:
conv_nodes = gcnn((ns, splits))
conv_nodes.shape

torch.Size([147, 255])

### XYTGATLayer

In [6]:
import torch
from torch import nn
from math import floor
from torch.nn import functional as F
from torch_geometric.nn.norm import GraphNorm
from math import floor
from model.tools import init_weights

In [ ]:
class XYTGATConv(nn.Module):
    @staticmethod
    def adj_mat(nodes):
        nodes = nodes.unsqueeze(-1)
        source = torch.tile(nodes, (1, 1, nodes.shape[1]))
        dest = source.transpose(1, 2)
        return torch.cat((source, dest), 0)

    def __init__(self,
                 in_channels,
                 out_channels,
                 att_heads,
                 xy_heads_ratio=0.2,
                 t_heads_ratio=0.1,
                 act_att=F.leaky_relu,
                 act_gat=F.elu,
                 norm=GraphNorm,
                 drop_prob=0.1,
                 bias=True,
                 w_init_f=lambda w: nn.init.xavier_uniform_(w),
                 device='cpu'):
        """A Graph Attention Convolution Layer
        param:act_att activation function for the attention scores
        param:act_gat activation function for the node update
        """
        super(XYTGATConv, self).__init__()
        self.device = device
        self.act_att = act_att
        self.act_gat = act_gat

        self.semantic_scores = nn.Conv2d(in_channels * 2,
                                    att_heads,
                                    1,
                                    device=device,
                                    bias=bias)

        xy_heads = max(1, floor((att_heads * xy_heads_ratio) + 0.5))
        self.xy_scores = nn.Conv2d(4,# x1,y1,x2,y2
                                   xy_heads,
                                   1,
                                   device=device,
                                   bias=bias)

        t_heads = max(1, floor((att_heads * t_heads_ratio) + 0.5))
        self.t_scores = nn.Conv2d(2,# t1,t2
                                  t_heads,
                                  1,
                                  device=device,
                                  bias=bias)

        self.node_update = nn.Conv1d((att_heads + xy_heads + t_heads) * in_channels,
                                     out_channels,
                                     1,
                                     device=device,
                                     bias=bias)

        self.norm = norm(in_channels).to(self.device)
        self.dropout = nn.Dropout(p=drop_prob) if drop_prob > 0.0 else nn.Identity()

        self.att_scores.apply(lambda m: init_weights(m, w_init_f))
        self.node_update.apply(lambda m: init_weights(m, w_init_f))
                

    def forward(self, x):
        pass

In [ ]:
class CausalXYTGATConv(XYTGATConv):
    def __init__(self,
                 in_channels,
                 out_channels,
                 att_heads,
                 xy_heads_ratio=0.2,
                 t_heads_ratio=0.1,
                 act_att=F.leaky_relu,
                 act_gat=F.elu,
                 norm=GraphNorm,
                 drop_prob=0.1,
                 bias=True,
                 w_init_f=lambda w: nn.init.xavier_uniform_(w),
                 device='cpu'):
        super().__init__(in_channels,
                         out_channels,
                         att_heads,
                         xy_heads_ratio,
                         t_heads_ratio,
                         act_att,
                         act_gat,
                         norm,
                         drop_prob,
                         bias,
                         w_init_f,
                         device)
    """A GATConv layer for a causal graph,
    where each node sees only itself and their predecessors
    """
    def forward(self, x):
        nodes, splits, xy_mats, t_mats = x
        # normalize graph
        batch_idx = torch.cat([torch.ones(s, dtype=torch.int) * i
                               for i, s in enumerate(splits)]).to(self.device)
        nodes = torch.transpose(nodes, 1, 0)
        nodes = self.norm(nodes, batch_idx.to(torch.long))
        nodes = torch.transpose(nodes, 1, 0)
        
        # generate adjacency matrix for whole batch
        subnodes = torch.split(nodes, splits, dim=1)
        subnodes = [self.dropout(sn) for sn in subnodes]
        adjs = [XYTGATConv.adj_mat(ns) for ns in subnodes]

        # generate raw semantic scores
        sem_raw_scores = [self.act_att(self.att_scores(adj)) for adj in adjs]
        
        # generate raw XY scores
        xy_raw_scores = [self.act_att(self.xy_scores(xy_mat)) for xy_mat in xy_mats]

        # generate raw T scores
        t_raw_scores = [self.act_att(self.t_scores(t_mat)) for t_mat in t_mats]
        
        print([s.shape for s in sem_raw_scores],
              [s.shape for s in xy_raw_scores],
              [s.shape for s in t_raw_scores],
              )
        return 
        rawscores = [torch.cat((sem, xy, t), dim=-1)
                     for sem, xy, t in zip(sem_raw_scores,
                                           xy_raw_scores,
                                           t_raw_scores)]        

        # set upper triangle to zero, so a node doesn't see future nodes
        adj_mask = [torch.tril(torch.ones(rs.shape,
                                dtype=torch.bool,
                                device=self.device))
                    for rs in rawscores]
        masked_scores = [torch.where(am,
                                     rs,
                                     torch.tensor(float('-inf'),
                                     device=self.device))
                         for am, rs in zip(adj_mask, rawscores)]

        # generate final att weights
        att_ws = [F.softmax(ms, dim=-1) for ms in masked_scores]

        # pass messages and generate output nodes
        gatconv = [torch.matmul(sn, s) for sn, s in zip(subnodes, att_ws)]
        gatconv = torch.cat(gatconv, -1)
        gatconv = gatconv.reshape((gatconv.shape[0] * gatconv.shape[1], gatconv.shape[2]))
        return self.act_gat(self.node_update(gatconv))

In [9]:
splits, fixations.shape

([47, 155, 53], torch.Size([255, 3, 1, 1]))

In [33]:
def all_pairs(t):
    t = t.unsqueeze(1)
    mat = torch.tile(t, (1, t.shape[0], 1))
    matT = torch.transpose(mat, 0, 1)
    return torch.cat((mat, matT), -1)

In [17]:
fixs = fixations.squeeze(-1).squeeze(-1)
fixs.shape

torch.Size([255, 3])

In [20]:
ts, xs, ys = torch.split(fixs, (1, 1, 1), dim=-1)
ts.shape, xs.shape, ys.shape

(torch.Size([255, 1]), torch.Size([255, 1]), torch.Size([255, 1]))

In [37]:
ts = torch.split(ts, splits, dim=0)
[t.shape for t in ts]

[torch.Size([47, 1]), torch.Size([155, 1]), torch.Size([53, 1])]

In [23]:
xys = torch.cat((xs, ys), -1)
xys = torch.split(xys, splits, dim=0)
[c.shape for c in xys]

[torch.Size([47, 2]), torch.Size([155, 2]), torch.Size([53, 2])]

In [36]:
xy_mats = [all_pairs(c) for c in xys]
[m.shape for m in xy_mats]

[torch.Size([47, 47, 4]), torch.Size([155, 155, 4]), torch.Size([53, 53, 4])]

In [38]:
t_mats = [all_pairs(t) for t in ts]
[m.shape for m in t_mats]

[torch.Size([47, 47, 2]), torch.Size([155, 155, 2]), torch.Size([53, 53, 2])]

In [39]:
scrs = [torch.cat((xy, t), -1) for xy, t in zip(xy_mats, t_mats)]
[s.shape for s in scrs]

[torch.Size([47, 47, 6]), torch.Size([155, 155, 6]), torch.Size([53, 53, 6])]

In [41]:
gcnn = CausalXYTGATConv(ns.shape[0],
                        ns.shape[0],
                        8,
                        device=conf.device)

In [42]:
gcnn

CausalXYTGATConv(
  (att_scores): Conv2d(294, 8, kernel_size=(1, 1), stride=(1, 1))
  (xy_scores): Conv2d(4, 2, kernel_size=(1, 1), stride=(1, 1))
  (t_scores): Conv2d(2, 1, kernel_size=(1, 1), stride=(1, 1))
  (node_update): Conv1d(1617, 147, kernel_size=(1,), stride=(1,))
  (norm): GraphNorm(147)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [43]:
conv_nodes = gcnn((ns, splits, xy_mats, t_mats))
conv_nodes.shape

RuntimeError: Given groups=1, weight of size [2, 4, 1, 1], expected input[1, 47, 47, 4] to have 4 channels, but got 47 channels instead

### GazeGrapher
<a id="grapher"></a>

GazeGrapher is a module that aggregates several GAT layers in sequence, performing all graph convolutions before graph readout and prediction.

PyTorch's native nn.Sequential does not keep residual tensors between its internal steps, so we need to extend it. Graph readout and predictions will need residual, intermediary, graphs

In [ ]:
from residual_sequential import ResidualSequential

We need input sizes for each of the graphs GAT Layers, and the residual layers' indices and GAT conv class of choice

In [ ]:
gat_channels = [conv_nodes.shape[0], 134, 134, 134, 67] # input size and 4 GAT convolution layers
residual_gat_levels = [False, True, False]
output_gat_levels = None
gat_conv = CausalGATConv

In [ ]:
from grapher import GazeGrapher

In [ ]:
grapher = GazeGrapher(channels=gat_channels,
                      output_gat_levels=output_gat_levels,
                      residual_gat_levels=residual_gat_levels,
                      device=conf.device)
grapher

In [ ]:
graphs = grapher((ns, splits)) # output is all selected residual graphs by residual_gat_levels
[g.shape for g in graphs]

### Readout
<a id="readout"></a>

Since we are dealing with complete causal graphs --each node sees ALL of its predecessors and itself-- we developed a causal graph readout. All information is aggregated by the last node at each convolution. So, we take the last node of each graph selected by residual_gat_levels, concatenate them, and run them through linear layers, outputting the final predictions.

In [ ]:
from readout import CausalReadout

In [ ]:
sum([g.shape[0] for g in graphs])

In [ ]:
ro_channels=conf.ro_channels
ro_channels[0] = sum([g.shape[0] for g in graphs])

In [ ]:
causal_ro = CausalReadout(channels=ro_channels,
                          dropout=conf.ro_dropout,
                          device=conf.device)

In [ ]:
causal_ro

In [ ]:
predictions = causal_ro((graphs, splits))
predictions.shape, labels.shape

### Whole pipeline
<a id="full"></a>

Since we now have all individual components, we are ready to run the full model.

In [ ]:
from gaze_only_gat import GazeOnlyGAT

In [ ]:
model = GazeOnlyGAT(feat_extractor=feat_ext,
                    gat_channels=gat_channels,
                    ro_hidden_features=ro_channels[1:],
                    device=conf.device)

In [ ]:
predictions = model((splits,
                     fixations.to(conf.device),
                     patches.to(conf.device),
                     labels.to(conf.device)))
predictions.shape, labels.shape